# 02 · Retrieval pipeline
Dense (Chroma cosine) + sparse (BM25) → Reciprocal Rank Fusion → rerank, always inside an **authorization filter**.
Requires notebook 01 to have been run (it reads the same store).

In [ ]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
from dotenv import load_dotenv
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# Automatically load environment variables from rag_service/.env and repo-root .env
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT.parent / ".env", override=False)

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env (default when keys are present)
has_keys = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY"))
OFFLINE = (os.getenv("NB_OFFLINE", "0" if has_keys else "1") == "1")
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings, reset_settings_cache
reset_settings_cache()
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

In [ ]:
from app.retriever import HybridRetriever, build_where
from app.store import get_store
assert get_store().count() > 0, "Run notebook 01 first"
def show(q, subj, k=3):
    r = HybridRetriever(where=build_where([subj]), top_k=k)
    print(f"Q: {q}   [scope={subj}]")
    for d, s in r.search(q): print(f"  {s:.3f} | {d.metadata['file_name']:<22} p.{d.metadata['page']} | {d.page_content[:70]!r}")
    print("  trace:", r.last_trace, "\n")
show("Which organelle produces ATP?", "BIO101")
show("When does total internal reflection occur?", "PHY101")

## LangChain retriever interface
`HybridRetriever` is a standard `BaseRetriever`, so it also works in any LangChain chain via `.invoke()`.

In [ ]:
r = HybridRetriever(where=build_where(["CS201"]), top_k=2)
for d in r.invoke("What is third normal form?"): print(d.metadata["file_name"], "|", d.page_content[:80])

## Authorization isolation: a Physics student must never see Biology chunks

In [ ]:
r = HybridRetriever(where=build_where(["PHY101"]), top_k=10)
hits = r.search("Which organelle produces ATP?")
print({d.metadata["subject_code"] for d, _ in hits})
assert all(d.metadata["subject_code"] == "PHY101" for d, _ in hits)

## Retrieval quality on the synthetic QA set (hit@3)

In [ ]:
import synthetic, pandas as pd
rows = []
for subj, q, kw, _ in synthetic.QA:
    top = HybridRetriever(where=build_where([subj]), top_k=3).search(q)
    rows.append({"subject": subj, "question": q, "hit@3": any(kw in d.page_content.lower() for d, _ in top)})
df = pd.DataFrame(rows); print("hit@3 =", df["hit@3"].mean()); df